# NGC 5010: a Thin Disk + Thick Disk + Spheroid Fit

The two-component model (`optimization_pipeline2_NGC5010_adam_lbfgs.ipynb`) reproduced NGC 5010's `V_LOS` map with the disk too weak: an antisymmetric residual of +-30 km/s confined to |z| < 1 kpc, the model rotating at 135 km/s in the midplane against an observed 158. That was not an optimizer failure. Every available parameter was swept at the fitted point and **the fit was already at the best the model family allows** -- raising `RsigZ` gave 117, cooling the disk radially gave 123, changing `Rd` gave 110, and `L0` changed nothing at all.

## What the measurements said

Decomposing the sightline at projected x = 2-3 kpc:

| | |
|---|---|
| `v_circ` at the light-weighted radius | 227 km/s |
| mean `v_phi` (after asymmetric drift) | 181 km/s |
| mean `v_LOS` (after projection) | **143 km/s** |
| observed | ~170 km/s |

The tracers at R = 2-3 kpc project to **169.4 km/s, essentially the observed value**. The model gets that physics right. But only 42% of the light on that sightline comes from R = 2-3; the rest comes from R > 3, where the projection factor collapses (R = 3-4 -> 142 km/s, R = 4-6 -> 103, R = 6-12 -> 73). The light-weighted average of that is 143.

So the disk has to be two things at once: concentrated enough that the midplane light is dominated by tracers near their tangent point, and extended enough to supply the light the galaxy actually shows at large radius and height. Measured on the observation, the model was too concentrated in projection (`<|x|>` 1.76 vs 1.95 kpc, `<|z|>` 0.52 vs 0.74) while still being too diluted along the midplane -- the two demands are not satisfiable by one disk DF. The spheroid could not take the second job either: raising `M_bulge` to 2e10 did lift midplane rotation 135 -> 145 and moved `<|x|>` and `<|z|>` the right way, but drove midplane `sigma` from 62 to 83 against an observed 54, because its light share and its temperature are locked together.

## What this notebook changes

One tracer population becomes two: a **thin disk** carrying the cold, fast-rotating midplane kinematics, and a **thick disk** carrying light at larger |x| and |z| at a higher dispersion, plus the spheroid as before. Both disks are sampled from the same `f_disc_from_params` DF with independent parameters, in the same Miyamoto-Nagai + Plummer + NFW potential. The observation this is aimed at is the vertical dispersion structure: `sigma` is 54 km/s in the midplane disk and 93 km/s at |z| = 1.5-3 kpc, a factor 1.7 that a single disk plus a compact spheroid could only fake by heating the disk everywhere.

**Nothing in `phoenix/` is modified** -- the sampler, loss and both optimizers are defined in this notebook, so the existing notebooks are unaffected.


In [ ]:
import os
import sys
import math
import numpy as np
import jax
import jax.numpy as jnp
import optax
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from jax.flatten_util import ravel_pytree

sys.path.insert(0, os.path.abspath('.'))
from geckos_data import load_geckos_maps

from phoenix.actions_to_phasespace.actions_to_phasespace_nn import PhoenixMapper
from phoenix.distribution_functions.sampling import sample_df_potential
from phoenix.distribution_functions.disk import f_disc_from_params
from phoenix.optimization.observables import spheroid_df_wrapper, bin_maps, blur_maps
from phoenix.optimization.pipeline import data_fit_loss, log_bounds_tree
from phoenix.potentials.potentials import (
    nfw_potential, plummer_potential, miyamoto_nagai_potential)


## 1. The Observed Galaxy

Distance and stellar mass from the GECKOS master catalogue, as in `fit_geckos.py`:
`kpc/arcsec = D[Mpc]*1e3/206265`, `Mstar = 10**lmstar`.

In [ ]:
NAME = 'NGC5010'
STEM = 'NGC5010_iDR2.0_1_SN100_5_MW_4800-8900'
DATADIR = os.path.join('data', '7_NGC5010', 'maps', STEM)

D_MPC = 42.2               # GECKOS master catalogue
LMSTAR = 10.6833
KPC_PER_ARCSEC = D_MPC * 1e3 / 206265.0
MSTAR = 10.0 ** LMSTAR

EXTENT_X, EXTENT_Z = 7.0, 4.0
GRID_SIZE = 40
CENTER_PERCENTILE = None    # well-centred pointing -> global flux centroid is correct
SPHEROID_COROTATION = 1.0   # rotation-dominated, no strongly hot core

obs_maps, info = load_geckos_maps(
    os.path.join(DATADIR, f'{STEM}_kin_maps.fits'),
    os.path.join(DATADIR, f'{STEM}_spatial_binning_maps.fits'),
    kpc_per_arcsec=KPC_PER_ARCSEC, extent_x=EXTENT_X, extent_z=EXTENT_Z,
    grid_size=GRID_SIZE, Mstar_fiducial=MSTAR, center_percentile=CENTER_PERCENTILE,
)

data_mask = np.array(obs_maps['mass']) > 0
vmax_mass = float(np.nanmax(np.array(obs_maps['mass'])))

print(f"{NAME}:  D = {D_MPC} Mpc -> {KPC_PER_ARCSEC:.4f} kpc/arcsec,  logM* = {LMSTAR}")
print(f"  PA = {info['position_angle_deg']:.1f} deg,  v_sys = {info['v_systemic_kms']:.1f} km/s")
print(f"  filled cells: {int(data_mask.sum())}/{GRID_SIZE**2} ({info['filled_fraction']*100:.0f}%)")
v, s = np.array(obs_maps['v_rot']), np.array(obs_maps['sigma'])
print(f"  v_rot in [{v[data_mask].min():.0f}, {v[data_mask].max():.0f}] km/s")
print(f"  sigma in [{s[data_mask].min():.0f}, {s[data_mask].max():.0f}] km/s")

# Quantify the two properties that set the modelling choices above.
_xc = 0.5 * (np.linspace(-EXTENT_X, EXTENT_X, GRID_SIZE + 1)[:-1]
             + np.linspace(-EXTENT_X, EXTENT_X, GRID_SIZE + 1)[1:])
_zc = 0.5 * (np.linspace(-EXTENT_Z, EXTENT_Z, GRID_SIZE + 1)[:-1]
             + np.linspace(-EXTENT_Z, EXTENT_Z, GRID_SIZE + 1)[1:])
_X, _Z = np.meshgrid(_xc, _zc)
_r = np.hypot(_X, _Z)
core, outer = (_r < 1.0) & data_mask, (_r > 2.5) & data_mask
print(f"  x-coverage [{_X[data_mask].min():+.1f}, {_X[data_mask].max():+.1f}] kpc "
      f"-> {abs(_X[data_mask].min() + _X[data_mask].max()) / np.ptp(_X[data_mask]) * 100:.0f}% off-centre")
print(f"  sigma core(r<1) = {s[core].mean():.0f} km/s,  outer(r>2.5) = {s[outer].mean():.0f} km/s"
      f"  -> ratio {s[core].mean() / s[outer].mean():.2f}")

In [ ]:
EXTENT = [-EXTENT_X, EXTENT_X, -EXTENT_Z, EXTENT_Z]
SIGMA_VMAX = 140.0     # NGC 5010 peaks near 112 km/s

def masked(a):
    return np.where(data_mask, np.array(a), np.nan)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
im = axes[0].imshow(masked(obs_maps['mass']), origin='lower', extent=EXTENT, cmap='magma',
                    norm=LogNorm(vmin=vmax_mass / 1e3, vmax=vmax_mass), aspect='auto')
axes[0].set_title('Observed light  (reference only, NOT fitted)')
fig.colorbar(im, ax=axes[0], label='fiducial M$_\\odot$')
im = axes[1].imshow(masked(obs_maps['v_rot']), origin='lower', extent=EXTENT,
                    cmap='seismic', vmin=-200, vmax=200, aspect='auto')
axes[1].set_title('Observed V$_{LOS}$'); fig.colorbar(im, ax=axes[1], label='km/s')
im = axes[2].imshow(masked(obs_maps['sigma']), origin='lower', extent=EXTENT,
                    cmap='viridis', vmin=0, vmax=SIGMA_VMAX, aspect='auto')
axes[2].set_title('Observed $\\sigma_{LOS}$'); fig.colorbar(im, ax=axes[2], label='km/s')
for ax in axes:
    ax.set_xlabel('x [kpc]'); ax.set_ylabel('z [kpc]')
plt.tight_layout(); plt.show()

## 2. The Three-Component Model

`sample_three_components` mirrors `observables.sample_and_map_particles`, with a second disk population. Four decisions in it are worth stating, because each one is a modelling choice rather than a mechanical extension:

- **One disk mass, split by `f_thin`.** The two disks share the potential's `M_disk`, divided by a single fitted fraction, rather than carrying two independent masses. This keeps the tracer budget consistent with the potential the DFs are sampled in and makes the split one interpretable number instead of a mass degeneracy.
- **The thin/thick labels are enforced by bounds, not discovered.** Two populations drawn from the same DF family are exchangeable: swapping their parameters gives an identical likelihood, so the fit has an exact label-switching degeneracy and, left alone, would wander between the two equivalent minima. The dispersion bounds are therefore made disjoint -- `thin` has `sigmaz0_R0` in (5, 35) and `thick` in (35, 150) -- which breaks the symmetry by construction. The consequence is that "thin" and "thick" mean *what those bounds say*, and the split is a prior, not a measurement.
- **The spheroid keeps `spheroid_corotation = 1.0`.** Measured on the previous fit, the spheroid already supplies 107 km/s of off-plane dispersion at full co-rotation against an observed 92.8, so this was never the limiting factor and there is no reason to change it here.
- **No Poisson term.** It is off for every real-galaxy fit in this repo (`w_poisson = 0`), and with two tracer disks in one Miyamoto-Nagai potential it would compare against a density the tracers are not asked to reproduce.

The optimizers are the same two stages as before -- log-space Adam with gradient clipping and bound projection, then projected L-BFGS with a Wolfe zoom line search at fixed bandwidth -- generalised to an arbitrary set of parameter groups.


In [ ]:
GROUPS = ('pot', 'thin', 'thick', 'bulge', 'mix')


# ---------------------------------------------------------------- log transform
def to_log(p):
    return {g: {k: jnp.log(jnp.asarray(v, dtype=jnp.float32)) for k, v in p[g].items()}
            for g in p}


def to_linear(params_log):
    return {g: {k: jnp.exp(v) for k, v in params_log[g].items()} for g in params_log}


# ---------------------------------------------------------------- sampling
def sample_three_components(mapper, pot, thin, thick, bulge, mix,
                            N_thin=12_000, N_thick=12_000, N_bulge=12_000,
                            prng_seed=0, spheroid_corotation=1.0):
    """Samples three tracer populations in one potential and maps them to phase space.

    Returns (x, y, z, vx, vy, vz, w) with the populations concatenated in the order
    thin, thick, bulge -- so a component's map is just a slice, exactly as the
    two-component version allows.

    The total disk tracer mass is `M_disk`, split between the two disks by `mix['f_thin']`.
    Tying them to one mass (rather than fitting two independent masses) keeps the tracer
    budget consistent with the potential that the DFs are sampled in, and leaves the
    thin/thick split as a single interpretable number.
    """
    M_halo, a_halo = pot['M_halo'], pot['a_halo']
    M_disk, a_disk, b_disk = pot['M_disk'], pot['a_disk'], pot['b_disk']
    M_bulge, a_bulge = pot['M_bulge'], pot['a_bulge']

    def total_potential(x, y, z):
        return (nfw_potential(x, y, z, M_halo, a_halo) +
                miyamoto_nagai_potential(x, y, z, M_disk, a_disk, b_disk) +
                plummer_potential(x, y, z, M_bulge, a_bulge))

    specs = [(f_disc_from_params, thin,  N_thin,  (100.0, 50.0, 3000.0)),
             (f_disc_from_params, thick, N_thick, (100.0, 50.0, 3000.0)),
             (spheroid_df_wrapper, bulge, N_bulge, (500.0, 500.0, 500.0))]

    key = jax.random.PRNGKey(prng_seed)
    cands, weights, angles = [], [], []
    for df, params, n, jb in specs:
        key, sk = jax.random.split(key)
        c, w = sample_df_potential(df=df, key=sk, params=params, Phi_xyz=total_potential,
                                   theta=(), n_candidates=n, envelope_max=None,
                                   J_bounds=jb, tau=0.05)
        key, sk = jax.random.split(key)
        cands.append(c)
        weights.append(w)
        angles.append(jax.random.uniform(sk, shape=(n, 3), minval=0.0, maxval=2 * jnp.pi))

    f_thin = mix['f_thin']
    masses = [f_thin * M_disk, (1.0 - f_thin) * M_disk, M_bulge]
    all_weights = jnp.concatenate([w / jnp.sum(w) * m for w, m in zip(weights, masses)])
    all_candidates = jnp.vstack(cands)
    all_angles = jnp.vstack(angles)

    N_tot = N_thin + N_thick + N_bulge
    nn_pot = jnp.array([M_halo / 1e11, a_halo, M_disk / 1e11, a_disk, b_disk,
                        M_bulge / 1e11, a_bulge])
    ps = mapper.map_to_phase_space(all_candidates, all_angles,
                                   jnp.tile(nn_pot, (N_tot, 1)))
    x, y, z = ps[:, 0], ps[:, 1], ps[:, 2]
    vx, vy, vz = ps[:, 3], ps[:, 4], ps[:, 5]

    R = jnp.maximum(jnp.sqrt(x**2 + y**2), 0.05)
    v_R = (x * vx + y * vy) / R
    v_phi = (x * vy - y * vx) / R

    # Prograde/retrograde assignment for the spheroid only (its DF is even in J_phi).
    key, sk = jax.random.split(key)
    u = jax.random.uniform(sk, shape=(N_bulge,))
    flip = jnp.where(u < spheroid_corotation, 1.0, -1.0)
    v_phi = v_phi.at[N_thin + N_thick:].multiply(flip)

    cos_phi, sin_phi = x / R, y / R
    return (x, y, z,
            v_R * cos_phi - v_phi * sin_phi,
            v_R * sin_phi + v_phi * cos_phi,
            vz, all_weights)


# ---------------------------------------------------------------- loss
def make_loss_fn_3c(mapper, obs_maps, N_thin, N_thick, N_bulge, grid_size,
                    extent_x, extent_z, prng_seed=0, loss_weights=(0.0, 3.0, 2.0),
                    spheroid_corotation=1.0, obs_bandwidth=None, reg_weight=0.0,
                    reg_center_log=None, mass_floor=1e-3):
    """`(loss, aux) = loss_fn(params_log, soft_bin_h)` for the three-component model.

    Mirrors `pipeline.make_loss_fn`: same `data_fit_loss`, same fixed observed-footprint
    mask taken from the unblurred maps, same matched blurring of the observation, same
    log-space Tikhonov prior. The Poisson self-consistency term is omitted -- it is off
    for every real-galaxy fit here (`w_poisson = 0`), and with two tracer disks sharing
    one Miyamoto-Nagai potential it would in any case compare against a density the
    tracers are not required to reproduce.
    """
    w_mass, w_vrot, w_sigma = loss_weights
    use_reg = reg_weight > 0 and reg_center_log is not None
    if use_reg:
        reg_vec, _ = ravel_pytree(reg_center_log)
    data_mask = obs_maps['mass'] > mass_floor

    def loss_fn(params_log, soft_bin_h=None):
        p = to_linear(params_log)
        x, y, z, vx, vy, vz, w = sample_three_components(
            mapper, p['pot'], p['thin'], p['thick'], p['bulge'], p['mix'],
            N_thin=N_thin, N_thick=N_thick, N_bulge=N_bulge,
            prng_seed=prng_seed, spheroid_corotation=spheroid_corotation)

        model_maps = bin_maps(x, z, vy, w, grid_size=grid_size, extent_x=extent_x,
                              extent_z=extent_z, soft_bin_h=soft_bin_h)
        target = obs_maps
        if obs_bandwidth is not None and soft_bin_h is not None:
            blur_h = jnp.sqrt(jnp.maximum(soft_bin_h**2 - obs_bandwidth**2, 0.0))
            target = blur_maps(obs_maps, blur_h, grid_size=grid_size,
                               extent_x=extent_x, extent_z=extent_z)

        mass_loss, vrot_loss, sigma_loss = data_fit_loss(
            model_maps, target, mass_floor=mass_floor, mask=data_mask)
        loss = w_mass * mass_loss + w_vrot * vrot_loss + w_sigma * sigma_loss

        reg = 0.0
        if use_reg:
            u_vec, _ = ravel_pytree(params_log)
            reg = reg_weight * jnp.mean((u_vec - reg_vec) ** 2)
            loss = loss + reg

        return loss, {'mass_loss': mass_loss, 'vrot_loss': vrot_loss,
                      'sigma_loss': sigma_loss, 'reg': reg}

    return loss_fn


# ---------------------------------------------------------------- optimizers
_HKEYS = ('loss', 'mass_loss', 'vrot_loss', 'sigma_loss', 'reg')


def _new_history():
    return {**{k: [] for k in _HKEYS}, 'params': []}


def _record(history, value, aux, params_log):
    history['loss'].append(float(value))
    for k in _HKEYS[1:]:
        history[k].append(float(aux[k]))
    lin = to_linear(params_log)
    history['params'].append({g: {k: float(v) for k, v in lin[g].items()} for g in lin})


def _result(params_log, history, **extra):
    lin = to_linear(params_log)
    return {'params': {g: {k: float(v) for k, v in lin[g].items()} for g in lin},
            'params_log': params_log, 'history': history, **extra}


def adam_fit(loss_fn, init_params_log, soft_bin_h, param_bounds, frozen_params=(),
             learning_rate=0.05, n_steps=600, grad_clip_norm=1.0, verbose_every=100):
    """Stage 1. Same construction as `pipeline.fit`: log-space Adam, global-norm gradient
    clipping, non-finite gradient sanitisation, frozen parameters zeroed, and a clip to
    the box bounds after every step. Fixed bandwidth (this model is not annealed)."""
    log_lo, log_hi = log_bounds_tree(init_params_log, param_bounds)
    h = None if soft_bin_h is None else jnp.asarray(soft_bin_h, jnp.float32)
    opt = optax.chain(optax.clip_by_global_norm(grad_clip_norm), optax.adam(learning_rate))

    @jax.jit
    def step(params_log, state):
        (value, aux), grads = jax.value_and_grad(loss_fn, has_aux=True)(params_log, h)
        grads = jax.tree_util.tree_map(lambda g: jnp.where(jnp.isfinite(g), g, 0.0), grads)
        if frozen_params:
            grads = {g: {k: (jnp.zeros_like(v) if k in frozen_params else v)
                         for k, v in grp.items()} for g, grp in grads.items()}
        updates, state = opt.update(grads, state, params_log)
        params_log = optax.apply_updates(params_log, updates)
        params_log = jax.tree_util.tree_map(jnp.clip, params_log, log_lo, log_hi)
        return params_log, state, value, aux

    params_log, state, history = init_params_log, None, _new_history()
    state = opt.init(params_log)
    for i in range(n_steps):
        params_log, state, value, aux = step(params_log, state)
        _record(history, value, aux, params_log)
        if verbose_every and (i % verbose_every == 0 or i == n_steps - 1):
            print(f"  adam {i:4d}   loss {float(value):.6f}")
    return _result(params_log, history)


def lbfgs_refine(loss_fn, init_params_log, soft_bin_h=None, param_bounds=None,
                 frozen_params=(), n_steps=100, memory_size=10,
                 max_linesearch_steps=30, grad_tol=1e-7, stall_tol=1e-9,
                 max_stalls=3, verbose_every=10):
    """Stage 2. Projected L-BFGS with a Wolfe zoom line search, at a FIXED bandwidth.

    Same three adaptations as the two-component notebook: frozen parameters are deleted
    from the optimised vector rather than gradient-zeroed (a zeroed coordinate still
    enters the curvature pairs and the line search), bounds are enforced by projection
    after each accepted step, and the objective is the identical closure Adam descended.
    The final history entry is evaluated at the returned parameters.
    """
    free = {g: {k: v for k, v in grp.items() if k not in frozen_params}
            for g, grp in init_params_log.items()}
    fixed = {g: {k: v for k, v in grp.items() if k in frozen_params}
             for g, grp in init_params_log.items()}
    log_lo, log_hi = log_bounds_tree(init_params_log, param_bounds)
    lo = {g: {k: v for k, v in grp.items() if k not in frozen_params}
          for g, grp in log_lo.items()}
    hi = {g: {k: v for k, v in grp.items() if k not in frozen_params}
          for g, grp in log_hi.items()}
    h = None if soft_bin_h is None else jnp.asarray(soft_bin_h, jnp.float32)

    merge = lambda fr: {g: {**fixed[g], **fr[g]} for g in init_params_log}
    loss_aux = lambda fr: loss_fn(merge(fr), h)
    value_fn = lambda fr: loss_aux(fr)[0]

    opt = optax.lbfgs(memory_size=memory_size,
                      linesearch=optax.scale_by_zoom_linesearch(
                          max_linesearch_steps=max_linesearch_steps))

    @jax.jit
    def step(fr, state):
        (value, aux), grad = jax.value_and_grad(loss_aux, has_aux=True)(fr)
        grad = jax.tree_util.tree_map(lambda g: jnp.where(jnp.isfinite(g), g, 0.0), grad)
        updates, state = opt.update(grad, state, fr, value=value, grad=grad,
                                    value_fn=value_fn)
        stepped = optax.apply_updates(fr, updates)
        clipped = jax.tree_util.tree_map(jnp.clip, stepped, lo, hi)
        proj = ravel_pytree(jax.tree_util.tree_map(
            lambda a, b: jnp.abs(a - b), stepped, clipped))[0].max()
        return clipped, state, value, aux, jnp.abs(ravel_pytree(grad)[0]).max(), proj

    evaluate = jax.jit(loss_aux)
    history = _new_history()
    state = opt.init(free)
    best_free, best_loss = free, np.inf
    stalls, n_projected, stop_reason, prev = 0, 0, f'reached n_steps={n_steps}', np.inf

    for i in range(n_steps):
        free, state, value, aux, gnorm, proj = step(free, state)
        value, gnorm = float(value), float(gnorm)
        _record(history, value, aux, merge(free))
        n_projected += int(proj > 0)
        if not np.isfinite(value):
            stop_reason = f'non-finite loss at iteration {i}'
            break
        if value < best_loss:
            best_loss, best_free = value, free
        if verbose_every and (i % verbose_every == 0 or i == n_steps - 1):
            print(f"  lbfgs {i:4d}   loss {value:.6f}   |grad|_inf {gnorm:.3e}")
        if gnorm < grad_tol:
            stop_reason = f'gradient below {grad_tol:g} at iteration {i}'
            break
        if abs(prev - value) < stall_tol * max(1.0, abs(prev)):
            stalls += 1
            if stalls >= max_stalls:
                stop_reason = f'no progress for {max_stalls} iterations (line search stalled)'
                break
        else:
            stalls = 0
        prev = value

    fv, fa = evaluate(best_free)
    _record(history, fv, fa, merge(best_free))
    if verbose_every:
        print(f"  stopped: {stop_reason}")
        if n_projected:
            print(f"  WARNING: bounds projection active on {n_projected} iterations")
    return _result(merge(best_free), history, n_iter=len(history['loss']) - 1,
                   stop_reason=stop_reason, n_projected=n_projected)


## 3. Configuration

In [ ]:
SEED = 0
OBS_BANDWIDTH = 0.4
N_PARTICLES = 12_000      # PER COMPONENT: 36,000 total, against 40,000 in the 2-component
N_PARTICLES_RENDER = 15_000   # notebook. Three populations cost memory in `bin_maps`,
N_STEPS = 600                 # whose intermediate is (grid_size^2 x N_total).
LEARNING_RATE = 0.05
N_LBFGS_STEPS = 100
LBFGS_MEMORY = 10
LOSS_WEIGHTS = (0.0, 3.0, 2.0)   # (mass, v_rot, sigma); the light map is not fitted
REG_WEIGHT = 0.01

# The potential starts from the converged two-component fit rather than from scratch:
# that fit's v_circ (227 km/s at R = 3.5) was if anything too high, so the mass
# distribution is not what needed fixing and re-deriving it wastes the run.
init = {
    'pot':   {'M_halo': 1.3e12, 'a_halo': 30.0, 'M_disk': 4.8e10, 'a_disk': 2.0,
              'b_disk': 0.26, 'M_bulge': 6.7e9, 'a_bulge': 0.7},
    # thin: cold and midplane-dominating -- this is the component that must carry the
    # 158 km/s midplane rotation, so it starts near the observed midplane sigma of 54.
    'thin':  {'R0': 4.0, 'Rd': 1.5, 'Sigma0': 1000.0, 'RsigR': 6.0, 'RsigZ': 6.0,
              'sigmaR0_R0': 30.0, 'sigmaz0_R0': 18.0, 'L0': 3.0, 'Rinit_for_Rc': 4.0},
    # thick: hot and extended -- aimed at the 93 km/s observed at |z| = 1.5-3 kpc.
    'thick': {'R0': 4.0, 'Rd': 3.0, 'Sigma0': 1000.0, 'RsigR': 8.0, 'RsigZ': 8.0,
              'sigmaR0_R0': 90.0, 'sigmaz0_R0': 60.0, 'L0': 3.0, 'Rinit_for_Rc': 4.0},
    'bulge': {'N0_spheroid': 5e9, 'J0_spheroid': 60.0, 'Gamma_spheroid': 1.5,
              'Beta_spheroid': 4.5, 'eta_spheroid': 1.0},
    'mix':   {'f_thin': 0.7},
}

_disk_common = {'R0': (1.0, 30.0), 'Rd': (0.5, 20.0), 'Sigma0': (1.0, 1e5),
                'RsigR': (0.5, 30.0), 'RsigZ': (0.5, 30.0),
                'L0': (0.5, 200.0), 'Rinit_for_Rc': (1.0, 30.0)}
param_bounds = {
    'pot':   {'M_halo': (1e9, 1e13), 'a_halo': (0.5, 100.0), 'M_disk': (1e8, 5e11),
              'a_disk': (0.2, 20.0), 'b_disk': (0.02, 3.0), 'M_bulge': (1e7, 5e11),
              'a_bulge': (0.05, 10.0)},
    # DISJOINT dispersion ranges: this is what breaks the thin/thick label degeneracy.
    'thin':  {**_disk_common, 'sigmaR0_R0': (5.0, 60.0),  'sigmaz0_R0': (5.0, 35.0)},
    'thick': {**_disk_common, 'sigmaR0_R0': (40.0, 200.0), 'sigmaz0_R0': (35.0, 150.0)},
    'bulge': {'N0_spheroid': (1e6, 1e13), 'J0_spheroid': (1.0, 2000.0),
              'Gamma_spheroid': (0.0, 2.8), 'Beta_spheroid': (3.2, 12.0),
              'eta_spheroid': (0.3, 5.0)},
    'mix':   {'f_thin': (0.05, 0.95)},
}

mapper = PhoenixMapper()
init_log = to_log(init)
print(f"{sum(len(g) for g in init.values())} parameters over {len(init)} groups")


### Which parameters can this dataset constrain?

In [ ]:
loss_fn = make_loss_fn_3c(
    mapper, obs_maps, N_thin=N_PARTICLES, N_thick=N_PARTICLES, N_bulge=N_PARTICLES,
    grid_size=GRID_SIZE, extent_x=EXTENT_X, extent_z=EXTENT_Z, prng_seed=SEED,
    loss_weights=LOSS_WEIGHTS, spheroid_corotation=SPHEROID_COROTATION,
    obs_bandwidth=OBS_BANDWIDTH, reg_weight=REG_WEIGHT, reg_center_log=init_log,
)

(_l, _a), grads = jax.value_and_grad(loss_fn, has_aux=True)(init_log, OBS_BANDWIDTH)
print(f"initial loss {float(_l):.5f}   "
      f"v_rot {float(_a['vrot_loss']):.5f}   sigma {float(_a['sigma_loss']):.5f}")

# Same freeze rule as the two-component notebook, and the same threshold: `L0` is inert
# because it enters only through rot = 0.5*(1 + tanh(Jphi/L0)) and this disk's Jphi is
# ~800 kpc km/s against L0 ~ 3, so the tanh is saturated for every tracer. `Sigma0` and
# `N0_spheroid` cancel exactly in the mass-renormalised weights.
FREEZE_GRAD_TOL = 1e-5
flat = [(abs(float(g)), grp, k) for grp in GROUPS for k, g in grads[grp].items()]
print("\n|d loss / d log(param)| at the initial guess:")
for mag, grp, k in sorted(flat, reverse=True):
    tag = '   <-- unconstrained: frozen' if mag < FREEZE_GRAD_TOL else ''
    print(f"  {grp:6s} {k:16s} {mag:.3e}{tag}")
FROZEN = tuple(sorted({k for mag, _, k in flat if mag < FREEZE_GRAD_TOL}))
print(f"\nfreezing: {FROZEN}")

n_par = sum(len(g) for g in init.values())
prior_scale = 2.0 * REG_WEIGHT / n_par
drifty = [(m, f'{g}.{k}') for m, g, k in flat if k not in FROZEN and m < prior_scale]
if drifty:
    print(f"\nweakly constrained (data gradient below the prior's {prior_scale:.2e} "
          f"restoring scale) -- free to drift, check AT BOUND in Section 8:")
    for m, nm in sorted(drifty):
        print(f"  {nm:22s} {m:.3e}")


## 4. Stage 1 - Adam

In [ ]:
result_adam = adam_fit(loss_fn, init_log, soft_bin_h=OBS_BANDWIDTH,
                       param_bounds=param_bounds, frozen_params=FROZEN,
                       learning_rate=LEARNING_RATE, n_steps=N_STEPS, verbose_every=100)
h_ad = result_adam['history']
print(f"\nADAM   loss {h_ad['loss'][0]:8.5f} -> {h_ad['loss'][-1]:8.5f}"
      + ('   [NaN!]' if not np.isfinite(h_ad['loss'][-1]) else ''))
print(f"       v_rot {h_ad['vrot_loss'][-1]:.5f}   sigma {h_ad['sigma_loss'][-1]:.5f}"
      f"   reg {h_ad['reg'][-1]:.5f}")


## 5. Stage 2 - L-BFGS Polish

In [ ]:
result_lbfgs = lbfgs_refine(loss_fn, result_adam['params_log'],
                            soft_bin_h=OBS_BANDWIDTH, param_bounds=param_bounds,
                            frozen_params=FROZEN, n_steps=N_LBFGS_STEPS,
                            memory_size=LBFGS_MEMORY, verbose_every=10)
h_lb = result_lbfgs['history']

# Handoff check: the first L-BFGS value is measured at Adam's final parameters, so it
# must match Adam's last recorded loss to within one Adam step (Adam records the loss
# BEFORE each update). A large gap would mean the two objectives are not the same.
print(f"\nobjective at the Adam endpoint: {h_lb['loss'][0]:.6f}"
      f"   (Adam's last recorded step: {h_ad['loss'][-1]:.6f})")
print(f"LBFGS  loss {h_lb['loss'][0]:8.5f} -> {h_lb['loss'][-1]:8.5f}"
      f"   (factor {h_lb['loss'][0] / max(h_lb['loss'][-1], 1e-30):.2f},"
      f" {result_lbfgs['n_iter']} iterations)")
print(f"       v_rot {h_lb['vrot_loss'][-1]:.5f}   sigma {h_lb['sigma_loss'][-1]:.5f}"
      f"   reg {h_lb['reg'][-1]:.5f}")
print(f"       stopped: {result_lbfgs['stop_reason']}")


In [ ]:
import json as _json
with open(f'{NAME}_fit_results_thin_thick_bulge.json', 'w') as _fh:
    _json.dump({
        'galaxy': NAME, 'model': 'thin disk + thick disk + spheroid',
        'config': {'D_MPC': D_MPC, 'LMSTAR': LMSTAR, 'GRID_SIZE': GRID_SIZE,
                   'EXTENT_X': EXTENT_X, 'EXTENT_Z': EXTENT_Z,
                   'N_PARTICLES_PER_COMPONENT': N_PARTICLES, 'N_STEPS': N_STEPS,
                   'N_LBFGS_STEPS': N_LBFGS_STEPS, 'OBS_BANDWIDTH': OBS_BANDWIDTH,
                   'LOSS_WEIGHTS': list(LOSS_WEIGHTS), 'REG_WEIGHT': REG_WEIGHT,
                   'FROZEN': list(FROZEN), 'SPHEROID_COROTATION': SPHEROID_COROTATION,
                   'CENTER_PERCENTILE': CENTER_PERCENTILE},
        'adam': {'params': result_adam['params'],
                 'final_loss': float(h_ad['loss'][-1])},
        'adam_lbfgs': {'params': result_lbfgs['params'],
                       'final_loss': float(h_lb['loss'][-1]),
                       'n_iter': result_lbfgs['n_iter'],
                       'stop_reason': result_lbfgs['stop_reason'],
                       'n_projected': result_lbfgs['n_projected']},
    }, _fh, indent=2)
print(f'wrote {NAME}_fit_results_thin_thick_bulge.json')


## 6. Convergence

In [ ]:
x_lb = np.arange(len(h_ad['loss']), len(h_ad['loss']) + len(h_lb['loss']))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(h_ad['loss'], color='tab:red', label='stage 1: Adam')
axes[0].plot(x_lb, h_lb['loss'], color='tab:blue', label='stage 2: L-BFGS')
axes[0].axvline(x_lb[0], color='gray', ls='--', lw=1, label='handoff')
axes[0].set_yscale('log'); axes[0].set_xlabel('iteration'); axes[0].set_ylabel('total loss')
axes[0].set_title(f'{NAME}: thin + thick + bulge'); axes[0].legend(fontsize=8)
for key, lab in [('vrot_loss', 'v_rot'), ('sigma_loss', 'sigma'), ('reg', 'prior')]:
    line, = axes[1].plot(h_ad[key], label=lab)
    axes[1].plot(x_lb, h_lb[key], color=line.get_color())
axes[1].axvline(x_lb[0], color='gray', ls='--', lw=1)
axes[1].set_yscale('log'); axes[1].set_xlabel('iteration'); axes[1].set_ylabel('loss component')
axes[1].set_title('Fitted components (light map not fitted)'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"total loss:  Adam {h_ad['loss'][-1]:.6f}  ->  +L-BFGS {h_lb['loss'][-1]:.6f}")
if result_lbfgs['n_projected']:
    print(f"bounds projection active on {result_lbfgs['n_projected']} iterations"
          f" -- check the AT BOUND flags in Section 8")


### Maps and residuals

In [ ]:
NR = N_PARTICLES_RENDER
p = result_lbfgs['params']
_x, _y, _z, _vx, _vy, _vz, _w = sample_three_components(
    mapper, p['pot'], p['thin'], p['thick'], p['bulge'], p['mix'],
    N_thin=NR, N_thick=NR, N_bulge=NR, prng_seed=SEED,
    spheroid_corotation=SPHEROID_COROTATION)

_bk = dict(grid_size=GRID_SIZE, extent_x=EXTENT_X, extent_z=EXTENT_Z,
           soft_bin_h=OBS_BANDWIDTH)
SLICES = {'thin': slice(0, NR), 'thick': slice(NR, 2 * NR), 'bulge': slice(2 * NR, 3 * NR)}
maps_total = bin_maps(_x, _z, _vy, _w, **_bk)
maps_comp = {n: bin_maps(_x[s], _z[s], _vy[s], _w[s], **_bk) for n, s in SLICES.items()}

specs = [('v_rot', 'seismic', dict(vmin=-200, vmax=200), 'V$_{LOS}$ (km/s)'),
         ('sigma', 'viridis', dict(vmin=0, vmax=SIGMA_VMAX), '$\\sigma_{LOS}$ (km/s)')]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for row, (key, cmap, kw, label) in enumerate(specs):
    for col, (t, arr) in enumerate([('Observed', obs_maps[key]),
                                    ('Model (3 components)', maps_total[key])]):
        im = axes[row, col].imshow(masked(arr), origin='lower', extent=EXTENT,
                                   cmap=cmap, aspect='auto', **kw)
        axes[row, col].set_title(f'{t}: {key}')
        fig.colorbar(im, ax=axes[row, col], label=label)
    d = np.where(data_mask, np.array(maps_total[key]) - np.array(obs_maps[key]), np.nan)
    rms = float(np.sqrt(np.nanmean(d ** 2)))
    im = axes[row, 2].imshow(d, origin='lower', extent=EXTENT, cmap='RdBu_r',
                             vmin=-60, vmax=60, aspect='auto')
    axes[row, 2].set_title(f'Model - Obs: {key}\n(RMS {rms:.1f} km/s)', fontsize=10)
    fig.colorbar(im, ax=axes[row, 2], label='model - obs (km/s)')
for ax in axes.ravel():
    ax.set_xlabel('x [kpc]'); ax.set_ylabel('z [kpc]')
fig.suptitle(f'{NAME}: thin + thick + bulge, kinematic-only fit', fontsize=13)
plt.tight_layout(); plt.show()

print(f"Residuals over the {int(data_mask.sum())} cells with data:")
for key in ('v_rot', 'sigma'):
    d = np.where(data_mask, np.array(maps_total[key]) - np.array(obs_maps[key]), np.nan)
    print(f"  {key:6s}: RMS {np.sqrt(np.nanmean(d**2)):6.2f} km/s   "
          f"median {np.nanmedian(d):+6.2f}   max|.| {np.nanmax(np.abs(d)):6.2f}")
print("\n  (two-component fit, for reference:  v_rot RMS 14.4,  sigma RMS 10.3 km/s)")


## 7. What Each Component Actually Does

This is the cell that says whether the extra freedom did the job it was added for. Three components can always fit better than two, so a lower loss proves nothing on its own; the question is whether the fit used them the way the diagnosis said it needed to -- a thin population dominating the midplane light and carrying the rotation, a thick one supplying light and dispersion off the plane -- or whether it simply found a second way to make one blended disk.

The failure mode to look for: if the thin and thick components end up with similar scale heights and similar light shares, the split is cosmetic and the dispersion bounds are the only thing keeping them apart.


In [ ]:
w_np = np.asarray(_w); z_np = np.asarray(_z); x_np = np.asarray(_x)
tot_w = w_np.sum()
xc = np.linspace(-EXTENT_X + EXTENT_X / GRID_SIZE, EXTENT_X - EXTENT_X / GRID_SIZE, GRID_SIZE)
zc = np.linspace(-EXTENT_Z + EXTENT_Z / GRID_SIZE, EXTENT_Z - EXTENT_Z / GRID_SIZE, GRID_SIZE)
Xg, Zg = np.meshgrid(xc, zc)
iz = int(np.argmin(np.abs(zc)))

print(f"{'component':10s} {'light share':>12s} {'rms z [kpc]':>12s} {'<|x|> [kpc]':>12s}")
for n, s in SLICES.items():
    ww, zz = w_np[s], z_np[s]
    m = np.where(data_mask, np.array(maps_comp[n]['mass']), 0.0)
    print(f"{n:10s} {ww.sum() / tot_w:12.3f} {np.sqrt((ww * zz**2).sum() / ww.sum()):12.2f}"
          f" {float(np.sum(m * np.abs(Xg)) / np.sum(m)):12.2f}")

M_tot = np.where(data_mask, np.array(maps_total['mass']), 0.0)
print(f"\nlight fraction by height (the structure the thick disk was added for):")
print(f"{'|z| band':>14s} {'thin':>8s} {'thick':>8s} {'bulge':>8s} {'obs sigma':>10s}"
      f" {'model sigma':>12s}")
for lo, hi in ((0, 0.5), (0.5, 1.0), (1.0, 1.5), (1.5, 2.5), (2.5, 4.0)):
    band = (np.abs(Zg) >= lo) & (np.abs(Zg) < hi) & data_mask
    if not band.any():
        continue
    tot = sum(np.where(data_mask, np.array(maps_comp[n]['mass']), 0.0)[band].sum()
              for n in SLICES)
    fr = [np.where(data_mask, np.array(maps_comp[n]['mass']), 0.0)[band].sum() / max(tot, 1e-30)
          for n in SLICES]
    print(f"  {lo:.1f}-{hi:.1f} kpc  {fr[0]:8.2f} {fr[1]:8.2f} {fr[2]:8.2f}"
          f" {np.median(np.array(obs_maps['sigma'])[band]):10.1f}"
          f" {np.median(np.array(maps_total['sigma'])[band]):12.1f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
row = data_mask[iz]
axes[0].plot(xc[row], np.array(obs_maps['v_rot'])[iz][row], 'k-', lw=2, label='observed')
axes[0].plot(xc[row], np.array(maps_total['v_rot'])[iz][row], color='tab:blue', label='model total')
for n, c in (('thin', 'tab:green'), ('thick', 'tab:orange'), ('bulge', 'tab:red')):
    axes[0].plot(xc[row], np.array(maps_comp[n]['v_rot'])[iz][row], ls='--', color=c, label=n)
axes[0].set_xlabel('x [kpc]'); axes[0].set_ylabel('V$_{LOS}$ [km/s]')
axes[0].set_title('midplane rotation, by component'); axes[0].legend(fontsize=8); axes[0].grid(alpha=.3)

zz_prof = [np.abs(zc) for _ in SLICES]
for n, c in (('thin', 'tab:green'), ('thick', 'tab:orange'), ('bulge', 'tab:red')):
    m = np.where(data_mask, np.array(maps_comp[n]['mass']), 0.0)
    prof = m.sum(axis=1) / np.maximum(M_tot.sum(axis=1), 1e-30)
    axes[1].plot(zc, prof, color=c, label=n)
axes[1].set_xlabel('z [kpc]'); axes[1].set_ylabel('share of the light')
axes[1].set_ylim(0, 1); axes[1].set_title('who supplies the light at each height')
axes[1].legend(fontsize=8); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

mid = (np.abs(Xg) > 1.0) & (np.abs(Xg) < 4.0) & (np.abs(Zg) < 0.5) & data_mask
off = (np.abs(Xg) < 4.5) & (np.abs(Zg) > 1.5) & (np.abs(Zg) < 3.0) & data_mask
print("\nthe two numbers this model was built to reconcile:")
for nm, sel in (('midplane  |x|=1-4, |z|<0.5', mid), ('off-plane |z|=1.5-3      ', off)):
    print(f"  {nm}:  |v| obs {np.median(np.abs(np.array(obs_maps['v_rot']))[sel]):6.1f}"
          f" model {np.median(np.abs(np.array(maps_total['v_rot']))[sel]):6.1f}"
          f"   |   sigma obs {np.median(np.array(obs_maps['sigma'])[sel]):5.1f}"
          f" model {np.median(np.array(maps_total['sigma'])[sel]):5.1f}")
print("  (two-component fit gave midplane |v| = 135.4 against an observed 157.8,"
      "\n   midplane sigma 62.1 against 54.1, off-plane sigma 94.7 against 92.8)")


## 8. Fitted Parameters

In [ ]:
import math
init_all = {f'{g}.{k}': v for g in GROUPS for k, v in init[g].items()}
fit_a = {f'{g}.{k}': v for g in GROUPS for k, v in result_adam['params'][g].items()}
fit_l = {f'{g}.{k}': v for g in GROUPS for k, v in result_lbfgs['params'][g].items()}
bnd = {f'{g}.{k}': v for g in GROUPS for k, v in param_bounds[g].items()}


def bound_flag(v, lo, hi, frac=0.02):
    if lo > 0 and v > 0 and hi > lo:
        f = (math.log(v) - math.log(lo)) / (math.log(hi) - math.log(lo))
    elif hi > lo:
        f = (v - lo) / (hi - lo)
    else:
        return ''
    return 'AT LOWER BOUND' if f <= frac else ('AT UPPER BOUND' if f >= 1 - frac else '')


print(f"{'parameter':22s} {'init':>11s} {'adam':>11s} {'+lbfgs':>11s} "
      f"{'fit/init':>9s} {'polish':>7s}  notes")
for g in GROUPS:
    print(f"-- {g}")
    for k in init[g]:
        nm = f'{g}.{k}'
        v, a, i0 = fit_l[nm], fit_a[nm], init_all[nm]
        notes = []
        if k in FROZEN:
            notes.append('frozen (unconstrained)')
        bf = bound_flag(v, *bnd[nm])
        if bf:
            notes.append(bf)
        polish = v / a if a else float('nan')
        if k not in FROZEN and abs(polish - 1.0) > 0.20:
            notes.append('MOVED BY POLISH -> Adam had not converged here')
        if k not in FROZEN and abs(v / i0 - 1.0) < 0.05 and REG_WEIGHT > 0:
            notes.append('~unmoved: set by the prior, not the data')
        print(f"{nm:22s} {i0:11.4g} {a:11.4g} {v:11.4g} {v / i0:9.2f} {polish:7.2f}"
              f"  {', '.join(notes)}")


## 9. Summary and Caveats

**The model:** a thin disk, a thick disk and a spheroid, all sampled in one Miyamoto-Nagai + Plummer + NFW potential, fitted to NGC 5010's `V_LOS` and `sigma_LOS` maps with `v_rot` weighted 3:2 over `sigma`, at a constant 0.4 kpc bandwidth, by Adam followed by a projected L-BFGS polish.

**The question this notebook exists to answer** is in Section 7, not in the loss: does the fit use the thin component to carry the midplane rotation and the thick one to carry the off-plane light and dispersion? A three-component model has ~10 more free parameters than a two-component one and will fit better regardless, so the loss improvement is not the result.

**Caveats.**

- **The thin/thick split is imposed, not measured.** Two populations from the same DF family are exchangeable, so the disjoint dispersion bounds are what make the labels mean anything. Read `sigmaz0_R0` sitting on the 35 km/s boundary between them as the fit trying to merge the two components back into one, and the split as a prior rather than a detection of a thick disk.
- **More parameters, same data.** The maps already failed to constrain the spheroid DF in the two-component fit (gradients ~1e-6). Adding a second disk does not add information, so expect several parameters to be prior-held or bound-pinned; the `AT BOUND`, `~unmoved` and drift flags are the check, and with `REG_WEIGHT = 0.01` the prior is not holding much.
- **No ground truth**, and the light map is still not fitted (`w_mass = 0`), so the radial light distribution -- the quantity that drives the projection dilution the whole diagnosis turned on -- remains unconstrained by data. Turning the mass term on was measured to push the model's light the wrong way for the two-component fit; whether that still holds with a thick disk available is an open question this notebook does not answer.
- **Float32** throughout, so a `line search stalled` stop reason in Section 5 is a precision limit rather than a bug.
- **Reduced tracer count:** 12,000 per component against 20,000 per component in the two-component notebook, to keep `bin_maps` memory in hand with three populations. Raise `N_PARTICLES` if the residual maps look grainy.
